In [1]:
%env MUJOCO_GL=egl
import math
import os
import pathlib
import sys
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".75"

import numpy as np
import mediapy as media

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config

sys.path.append('../py_script')

env: MUJOCO_GL=egl


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /work/11430/jpeng303/vista/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


In [2]:
def _quat2axisangle(quat):
    """
    Copied from robosuite: https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py#L490C1-L512C55
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den
    
# Load pi model
def create_pi05(model_name, checkpoint_dir=None, assets_dir=None):
    vla_config = _config.get_config(model_name)
    if checkpoint_dir is None:
        checkpoint_dir = download.maybe_download(f"gs://openpi-assets/checkpoints/{model_name}")
    else:
        checkpoint_dir = pathlib.Path(checkpoint_dir).resolve()

    # Load norm stats from assets dir if checkpoint doesn't have them
    norm_stats = None
    if assets_dir is not None:
        data_config = vla_config.data.create(vla_config.assets_dirs, vla_config.model)
        if data_config.asset_id is not None:
            from openpi.training import checkpoints as _checkpoints
            assets_path = pathlib.Path(assets_dir).resolve()
            norm_stats = _checkpoints.load_norm_stats(assets_path, data_config.asset_id)

    initial_scene_plan = {
        "Plan: TBD.\n"
        " What I have done: TBD.\n"  # Extra space here follows the original dataset formatting...
        "Now I need to do: TBD.\n"
    }
    policy = _policy_config.create_trained_skill_reasoning_policy(
        vla_config,
        checkpoint_dir,
        norm_stats=norm_stats,
        initial_scene_plan=initial_scene_plan,
        sample_kwargs={
            "temperature": 0.0,
            "max_reasoning_steps": 256,
            "force_initial_reasoning": True
        }
    )
    return policy

def prompt_from_obs(obs, task, scene_plan='', skill='', mode='thinking'):
    """Build observation dict for Pi0Fuse inference.

    The thought prefix must match the training format from cot_simple.json:
      "Instruction: <task>\\n<scene_plan>"
    A 1-element thought list triggers action-mode tokenization (BEGIN_OF_ACTION
    suffix), which prefill() strips before deciding to think or act.
    """
    state_vec = np.concatenate(
        (
            obs["robot0_eef_pos"],
            _quat2axisangle(obs["robot0_eef_quat"]),
            obs["robot0_gripper_qpos"],
        )
    )

    print("current scene plan: ", scene_plan)

    if mode == 'thinking':
        if scene_plan == '':
            thought_prefix = task
        else:
            thought_prefix = scene_plan
        text_input = thought_prefix

    else:
        print(f"Skill: {skill}")
        text_input = skill

    return {
        'observation/image': obs['agentview_image'][::-1, ::-1, :],
        'observation/wrist_image': obs['robot0_eye_in_hand_image'][::-1, ::-1, :],
        "observation/state": state_vec,
        'prompt': task,
        'thought': [text_input],
        'mode': mode,
    }


In [4]:
import importlib
import openpi
importlib.reload(openpi.models.gemma)
from openpi.policies import policy_config as _policy_config

# model_name = 'pi05_libero_skill_reason_lora_v2'
model_name = 'pi05_libero_skill_reason_fixed'
# Create a trained policy.
policy = create_pi05(   # Use LoRA weights:
    model_name,
    # checkpoint_dir='../pace/openpi/checkpoints/pi05_libero_reason_lora/pi05_libero_reason_lora/7999',
    checkpoint_dir=f'../pace/openpi/checkpoints/{model_name}/{model_name}/20000',
    assets_dir=f'../pace/openpi/assets/{model_name}'
    # 'pi05_libero_10_reason_lora',
    # checkpoint_dir='../pace/openpi/checkpoints/pi05_libero_10_reason_lora/pi05_libero_10_reason_lora/1500',
    # assets_dir='../pace/openpi/assets/pi05_libero_10_reason_lora'
)

/work/11430/jpeng303/vista/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/jax/extend/linear_util.py:38: DeprecationWarning: linear_util.wrap_init is missing a DebugInfo object. This behavior is deprecated, use api_util.debug_info() to construct a proper DebugInfo object and propagate it to this function. See https://github.com/jax-ml/jax/issues/26480 for more details.
  debug_info = debug_info or _missing_debug_info("linear_util.wrap_init")
/work/11430/jpeng303/vista/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/jax/extend/linear_util.py:38: DeprecationWarning: linear_util.wrap_init is missing a DebugInfo object. This behavior is deprecated, use api_util.debug_info() to construct a proper DebugInfo object and propagate it to this function. See https://github.com/jax-ml/jax/issues/26480 for more details.
  debug_info = debug_info or _missing_debug_info("linear_util.wrap_init")
/work/11430/jpeng303/vista/workspace/mujoco_test/mujoc

In [5]:
def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_depths": True}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

class LiberoEnvMaker:
    def __init__(self, suite: str,
                 render_resolution: int = 512, seed: int = 0,
                 repeats: int = 1):
        benchmark_dict = benchmark.get_benchmark_dict()
        self.task_suite = benchmark_dict[suite]()
        self.repeats = repeats
        self.render_resolution = render_resolution
        self.seed = seed

    def get_num_tasks(self):
        return self.task_suite.n_tasks

    def task_instantiations(self, task_id):
        task = self.task_suite.get_task(task_id)
        print(task, task_id)
        initial_states = self.task_suite.get_task_init_states(task_id)
        env, task_description = _get_libero_env(task, self.render_resolution, self.seed)
        for episode_idx in range(self.repeats):
            env.reset()
            obs = env.set_init_state(initial_states[episode_idx])
            yield obs, env, task_description


In [6]:
import mujoco

def add_visual_point(scn, pos, rgba=[1, 0, 0, 1], radius=0.01):
    """Add a sphere marker at a 3D position to an MjvScene.
    
    Args:
    scn: mujoco.MjvScene object (e.g. env.sim._render_context_offscreen.scn)
    pos: [x, y, z] position
    rgba: [r, g, b, a] color
    radius: sphere radius
    """
    if scn.ngeom >= scn.maxgeom:
        print("WARNING: scene buffer full!")
        return  # scene buffer full
    
    mujoco.mjv_initGeom(
        scn.geoms[scn.ngeom],
        type=mujoco.mjtGeom.mjGEOM_SPHERE,
        size=[radius, 0, 0],
        pos=np.array(pos, dtype=np.float64),
        mat=np.eye(3, dtype=np.float64).flatten(),
        rgba=np.array(rgba, dtype=np.float32),
    )
    scn.ngeom += 1

def patch_env_for_render(env):
    """
    Patch the rendering function of a robosuite environment to add the ability to draw 3d points.
    This code was generated by Claude
    """
    # Get the render context
    render_ctx = env.sim._render_context_offscreen
    # Save original render method (unused)
    original_render = env.sim.render

    def render_with_points(*args, visual_points=[], **kwargs):
        env._visual_points = visual_points
        result = original_render(*args, **kwargs)
        env._visual_points = []
        return result
    env.sim.render = render_with_points

    def patched_render(width, height, camera_id=None, segmentation=False):
        """
        Copy of original function, but we needed to break it up.

        visual_points should be a list of dicts: {"pos": [x,y,z], "rgba": [r,g,b,a], "radius": 0.01}
        """
        # Call original up to mjv_updateScene (we need to replicate the logic)
        viewport = mujoco.MjrRect(0, 0, width, height)
        
        if width > render_ctx.con.offWidth or height > render_ctx.con.offHeight:
            new_width = max(width, render_ctx.model.vis.global_.offwidth)
            new_height = max(height, render_ctx.model.vis.global_.offheight)
            render_ctx.update_offscreen_size(new_width, new_height)
        
        if camera_id is not None:
            if camera_id == -1:
                render_ctx.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
            else:
                render_ctx.cam.type = mujoco.mjtCamera.mjCAMERA_FIXED
                render_ctx.cam.fixedcamid = camera_id
        
        mujoco.mjv_updateScene(
            render_ctx.model._model, render_ctx.data._data,
            render_ctx.vopt, render_ctx.pert, render_ctx.cam,
            mujoco.mjtCatBit.mjCAT_ALL, render_ctx.scn
        )
        
        if segmentation:
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_SEGMENT] = 1
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_IDCOLOR] = 1
        
        # --- Add visual points here ---
        for pt in env._visual_points:
            add_visual_point(render_ctx.scn, pt["pos"], pt.get("rgba", [1,0,0,1]), pt.get("radius", 0.01))
        
        mujoco.mjr_render(viewport=viewport, scn=render_ctx.scn, con=render_ctx.con)
        
        if segmentation:
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_SEGMENT] = 0
            render_ctx.scn.flags[mujoco.mjtRndFlag.mjRND_IDCOLOR] = 0

    render_ctx.render = patched_render

def get_cam_pose(env, cam_name="agentview"):
    cam_id = env.sim.model.camera_name2id(cam_name)
    return (env.sim.model.cam_pos[cam_id], env.sim.model.cam_quat[cam_id])
    
def set_cam_pose(env, pose, cam_name="agentview"):
    cam_id = env.sim.model.camera_name2id(cam_name)
    env.sim.model.cam_pos[cam_id] = pose[0]
    env.sim.model.cam_quat[cam_id] = pose[1]

In [7]:
sys.path.append('../datasets')
from probe_network import ProbeNetwork
import orbax.checkpoint as ocp
import flax.nnx as nnx
checkpointer = ocp.StandardCheckpointer()
checkpoint_dir = os.path.abspath('../datasets/checkpoints/state')
probe = ProbeNetwork(nnx.Rngs(0))
graphdef, abstract_state = nnx.split(probe)
state_restored = checkpointer.restore(checkpoint_dir, abstract_state)
probe = nnx.merge(graphdef, state_restored)

FileNotFoundError: Checkpoint at /work/11430/jpeng303/vista/workspace/mujoco_test/datasets/checkpoints/state not found.

In [16]:
rollout = []
frames = []
wrist_frames = []
snapshots = []
statuses = []
subtasks = []
explain_frames = []

libero_envs = LiberoEnvMaker('libero_10')

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [19]:
# STARTUP section
# instance = next(libero_envs.task_instantiations(3))
# obs, env, task_description = instance
# patch_env_for_render(env)
# policy.start()
# print(task_description)
# scene_plan = '1. PICKUP_FROM(bowl, table) 2. PLACE_IN(bowl, bottom drawer) 3. CLOSE(bottom drawer)'
# prompt = prompt_from_obs(obs, scene_plan, mode='thinking')
# vla_output = policy.infer(prompt)
# skill = vla_output['subtask']

#task_description = "Place the bowl into the top drawer."
# prev_pos, prev_quat = save_cam_pose
# set_cam_pose(env, (prev_pos+[0, 0, 1], prev_quat))
# obs, reward, done, info = env.step(np.zeros(7))
# break
#skill = 'PICK(bowl)'
#scene_plan = ''
#scene_plan = 'Plan: 1. Open the top drawer. \
#What have I done: Open the top drawer. \
#Now I need to do: Nothing. Task complete.'
#prompt = prompt_from_obs(obs, task_description, skill=skill, mode='acting')
prompt = prompt_from_obs(obs, scene_plan, skill=skill, mode='acting')
vla_output = policy.infer(prompt)
actions = vla_output['actions']
subtask = vla_output.get('subtask', None)
rollout.append(obs)

trajectory_idx = 0
for i in range(50):
    act = np.copy(actions[trajectory_idx])
    obs, reward, done, info = env.step(act)
    rollout.append(obs)
    frames.append(obs['agentview_image'][::-1, ::-1, :])
    wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])
    trajectory_idx += 1
    if trajectory_idx == (len(actions)//2):
        prompt = prompt_from_obs(obs, task_description, skill=skill, mode='acting')
        vla_output = policy.infer(prompt)
        intermediates = policy.saved_intermediates
        # First element is the reasoning side output
        # (Add batch dimension, take all layers, remove old batch dimension, take last token, take full embedding)[remove batch dimension]
        pred_transform = probe(intermediates[0][None, :, 0, -1, :])[0]
        actions = vla_output['actions']
        points = [{'pos': pred_transform + obs['robot0_eef_pos']}]
        print(points)
        explain_img = env.sim.render(width=512, height=512, camera_name="agentview", visual_points=points)[::-1, ::-1, :]
        explain_frames.append(explain_img)
        trajectory_idx = 0

freq = 20
#save_cam_pose = get_cam_pose(env)
print(task_description)
img = obs['agentview_image'][::-1, ::-1, :]
import matplotlib.pyplot as plt
plt.figure(0)
plt.clf()
plt.imshow(img)
plt.figure(1)
plt.clf()
plt.imshow(obs['agentview_depth'][::-1, ::-1, :])
plt.show()
media.write_video(f'franka_90.mp4', frames, fps=freq)
media.write_video(f'franka_explain.mp4', explain_frames, fps=2)
media.write_video(f'franka_90_wrist.mp4', wrist_frames, fps=freq)

current scene plan:  
Skill: CLOSE(bottom drawer)
inputs -------------------> ['CLOSE(bottom drawer)']
mode acting now...
current scene plan:  
Skill: CLOSE(bottom drawer)
inputs -------------------> ['CLOSE(bottom drawer)']
mode acting now...
[{'pos': Array([ -1.925198, -10.777014,  10.271869], dtype=float32)}]
current scene plan:  
Skill: CLOSE(bottom drawer)
inputs -------------------> ['CLOSE(bottom drawer)']
mode acting now...
[{'pos': Array([ -2.13595 , -10.983439,  10.29744 ], dtype=float32)}]
current scene plan:  
Skill: CLOSE(bottom drawer)
inputs -------------------> ['CLOSE(bottom drawer)']
mode acting now...
[{'pos': Array([ -2.1297083, -11.268285 ,  10.308855 ], dtype=float32)}]
current scene plan:  
Skill: CLOSE(bottom drawer)
inputs -------------------> ['CLOSE(bottom drawer)']
mode acting now...
[{'pos': Array([ -2.3718557, -11.042945 ,  10.258924 ], dtype=float32)}]
current scene plan:  
Skill: CLOSE(bottom drawer)
inputs -------------------> ['CLOSE(bottom drawer)']
m

In [ ]:
def get_info(o):
    print(type(o), dir(o))

In [ ]:
instance = next(libero_envs.task_instantiations(0))
obs, env, task_description = instance
patch_env_for_render(env)

points = [{'pos': obs['robot0_eef_pos']}]
img = env.sim.render(width=512, height=512, camera_name="agentview", visual_points=points)[::-1, ::-1, :]
plt.figure(0)
plt.clf()
plt.imshow(img)

In [ ]:
#frames = []
#wrist_frames = []
#instance = next(libero_envs.task_instantiations(0))
#obs, env, task_description = instance

trajectory_idx = 0
for i in range(50):
    act = [0, 0.1, 0, 0, 0, 0, 0]
    obs, reward, done, info = env.step(act)
    frames.append(obs['agentview_image'][::-1, ::-1, :])
    wrist_frames.append(obs['robot0_eye_in_hand_image'][::-1, ::-1, :])

freq = 20
#save_cam_pose = get_cam_pose(env)
print(task_description)
img = obs['agentview_image'][::-1, ::-1, :]
import matplotlib.pyplot as plt
plt.figure(0)
plt.clf()
plt.imshow(img)
plt.figure(1)
plt.clf()
plt.imshow(obs['agentview_depth'][::-1, ::-1, :])
plt.show()
media.write_video(f'franka_90.mp4', frames, fps=freq)
media.write_video(f'franka_90_wrist.mp4', wrist_frames, fps=freq)

In [ ]:
instance = next(libero_envs.task_instantiations(0))
obs, env, task_description = instance
policy.start()